# Week 4 Lab: Neural Networks from Scratch

Welcome to the Week 4 Lab of AI Hands-On! In our lecture, we discussed how Deep Learning is a subset of Machine Learning that uses neural networks with many layers ("deep" networks). 

Today, we are taking away the "magic" of libraries like PyTorch or TensorFlow. We will build a 2-layer Neural Network from the ground up using only simple mathematics and `numpy`. We will use a subset of the famous MNIST dataset (images of handwritten digits) to train our model to recognize numbers.

### Step 1: Initialization and Data Setup
First, we import our foundational libraries. `numpy` is crucial for the heavy linear algebra, and `matplotlib` will help us visualize our results.

![Preview of MNIST Digits](/Users/cs/Desktop/AI_HandsOn_Course/Images/MNIST_dataset_example.png)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml


mnist = fetch_openml('mnist_784', version =1, parser = 'auto')

X_full = mnist.data.values.T / 255

Y_full = mnist.target.values.astype(int)

X_train = X_full[:, :1000]
Y_train = Y_full[:1000]

print(X_train.shape)
print(Y_train.shape)
print(X_train[150:170,0])


### Step 2: The Math of a Neuron and Activation Functions

As we learned in theory, a single perceptron calculates a linear combination of inputs and passes it through a non-linear activation function. The bias is an additional parameter in the perceptron model that allows you to shift the activation function to the left or right.

For our hidden layer, the pre-activation value $Z$ is calculated as:
$$Z^{[1]} = W^{[1]} X + b^{[1]}$$

To allow our Neural Network to deal with non-linear data, we apply a non-linear activation function. We will use the Rectified Linear Unit (ReLU), mathematically defined as:
$$g(z) = \max(0, z)$$

For our final output layer, we want probabilities for each of the 10 digit classes. We will use the **Softmax** function, which turns our raw scores into probabilities that sum to 1.

![Perceptron Diagram](/Users/cs/Desktop/AI_HandsOn_Course/Images/Perceptron.png)



In [ ]:
def relu(Z):
    return np.maximum(0, Z)

def softmax(Z):
    A = np.exp(Z) / sum(np.exp(Z))
    return A

### Step 3: Forward Propagation

Forward propagation is the process of taking an input image and running it through the network to get a prediction. 

Our architecture:
1.  **Input Layer:** 784 nodes ($28 \times 28$ pixels)
2.  **Hidden Layer:** 10 nodes (using ReLU)
3.  **Output Layer:** 10 nodes (using Softmax)

We first need to initialize our weights ($W$) and biases ($b$) randomly. Then, the forward pass equations are:
$$Z^{[1]} = W^{[1]} X + b^{[1]}$$
$$A^{[1]} = \text{ReLU}(Z^{[1]})$$
$$Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}$$
$$A^{[2]} = \text{Softmax}(Z^{[2]})$$

![NN Diagram](/Users/cs/Desktop/AI_HandsOn_Course/Images/NeuralNetwork.png)

In [ ]:
def init_params():
    W1 = np.random.rand(10, 784) - 0.5 
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5 
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = relu(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2


### Step 4: Quantifying the Error

To measure how well the Neural Network's prediction match the ground truth, we need an empirical loss function. This loss guides the optimization process during training, and the goal is to minimize it.

Before we can compare our prediction probabilities ($A^{[2]}$) to the actual labels ($Y$), we must convert the labels into a format the math understands. If the label is `4`, we don't subtract 4. Instead, we use **One-Hot Encoding** to create an array where the 4th index is `1` and the rest are `0`.

In [ ]:
def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T 
    return one_hot_Y


### Step 5: Backpropagation & Gradient Descent

This is where the network actually learns! Training Neural Networks involves adjusting weights to minimize an objective function (loss function). 

We will use Gradient Descent, which iteratively updates the variables in the direction of the steepest decrease. To do this, we use Backpropagation—an algorithm that efficiently computes gradients using the chain rule. 

We compute the error at the output layer ($dZ^{[2]}$) and propagate it backwards:
$$dZ^{[2]} = A^{[2]} - Y_{\text{one\_hot}}$$
$$dW^{[2]} = \frac{1}{m} dZ^{[2]} A^{[1]T}$$
$$db^{[2]} = \frac{1}{m} \sum dZ^{[2]}$$

We apply the reverse operations for the hidden layer, utilizing the derivative of our ReLU function. Finally, we update our parameters using the learning rate ($\alpha$):
$$W \leftarrow W - \eta dW$$

In [ ]:
def deriv_relu(Z):
    return Z > 0 

def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    m = Y.size
    one_hot_Y = one_hot(Y)

    dZ2 = A2 - one_hot_Y
    dW2 = 1/m * dZ2.dot(A1.T)
    db2 = 1/m * np.sum(dZ2)

    dZ1 = W2.T.dot(dZ2) * deriv_relu(Z1)
    dW1 = 1/m * dZ1.dot(X.T)
    db1 = 1/m * np.sum(dZ1)

    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2

def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    return np.sum(predictions == Y) / Y.size


def gradient_descent(X, Y, alpha, iterations):
    W1, b1, W2, b2 = init_params()
    
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)

        if i % 10 == 0:
            print(f"Iteration: {i}")
            predictions = get_predictions(A2)
            print(f"Accuracy: {get_accuracy(predictions, Y):.4f}")

    return W1, b1, W2, b2

W1, b1, W2, b2 = gradient_descent(X_train, Y_train, alpha = 0.1, iterations = 500)





In [ ]:
# 2. Prepare Test Set
# We select 20 images the network has never seen (from index 1000 to 1020)
X_test = X_full[:, 1000:1020]
Y_test = Y_full[1000:1020]

# 3. Perform inference on new data
_, _, _, A2_test = forward_prop(W1, b1, W2, b2, X_test)
test_predictions = get_predictions(A2_test)

# 4. Visualize 10 random predictions
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
axes = axes.flatten()

for i in range(10):
    # Pick a random index from our test subset
    idx = np.random.randint(0, X_test.shape[1])
    
    # Reshape the flattened 784 vector back to 28x28 for display
    image = X_test[:, idx].reshape(28, 28)
    true_label = Y_test[idx]
    pred_label = test_predictions[idx]
    
    # Display the image
    axes[i].imshow(image, cmap='gray')
    axes[i].axis('off')
    
    # Use green title if correct, red if incorrect
    color = 'green' if true_label == pred_label else 'red'
    axes[i].set_title(f"Pred: {pred_label} | True: {true_label}", color=color)

plt.tight_layout()
plt.show()